# Sprint 2: Configuración y splits (4 clases)

Este notebook hace:
1. Cuenta Infiltration en el subset (lo que faltó en el notebook anterior).
2. Escribe `configs/sprint2.yaml` en Drive.
3. Escribe `scripts/prepare_splits_s2.py` en Drive.
4. Ejecuta el script para generar los nuevos splits.
5. Verifica la distribución de las 4 clases.

**Runtime:** CPU.


## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)


## 2. Contar Infiltration (dato faltante)

In [ ]:
import pandas as pd
import os

NIH_DIR = f'{PROJECT_ROOT}/data/raw/nih'
df = pd.read_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv')

print(f'Total imágenes en el subset: {len(df):,}')
print()
print('DISTRIBUCIÓN COMPLETA DE LAS 4 CLASES:')
print('=' * 55)
for cls in ['No Finding', 'Cardiomegaly', 'Effusion', 'Infiltration']:
    if cls == 'No Finding':
        n = (df['Finding Labels'] == 'No Finding').sum()
    else:
        n = df['Finding Labels'].str.contains(cls).sum()
        n_pure = (df['Finding Labels'] == cls).sum()
        print(f'{cls} (combinada):  {n:>6,}')
        print(f'  - {cls} pura:    {n_pure:>6,}')
        continue
    print(f'{cls}:           {n:>6,}')

print()
print('CLASE MÁS PEQUEÑA (define el techo del balance):')
positives = {}
for cls in ['Cardiomegaly', 'Effusion', 'Infiltration']:
    positives[cls] = df['Finding Labels'].str.contains(cls).sum()
ceiling = min(positives.values())
print(f'  {min(positives, key=positives.get)}: {ceiling:,}')
print()
print(f'Dataset estimado Sprint 2:')
print(f'  {ceiling} x 3 clases positivas = {ceiling*3:,} positivas')
print(f'  {ceiling*2} No Finding (ratio 2:1)')
print(f'  Total: ~{ceiling*5:,} imágenes (antes de 1 img/paciente)')


## 3. Escribir `configs/sprint2.yaml`

In [ ]:
from pathlib import Path

content = r'''# configs/sprint2.yaml
# Sprint 2 - Baseline CNN optimizado, 4 clases, finetune full
# ─────────────────────────────────────────────────────────────
# Clases:
#   0 - No Finding
#   1 - Cardiomegaly   (insuficiencia cardíaca)
#   2 - Infiltration   (TB pulmonar / neumonía) ← más relevante en Loayza
#   3 - Effusion       (derrame pleural)
#
# Justificación clínica (Hospital Nacional Arzobispo Loayza):
#   Las 3 clases positivas cubren las patologías más frecuentes en el
#   servicio de Neumología / Medicina Interna del HNAL, donde la TB,
#   la neumonía y la insuficiencia cardíaca representan la mayor carga
#   asistencial (MINSA 2024, Lima concentra 44% de casos TB nacionales).
#
# Diferencias respecto a Sprint 1 (baseline.yaml):
#   - num_classes: 2 → 4
#   - finetune_mode: "head" → "full"
#   - learning_rate: 0.001 → 0.00001  (10x más bajo para full fine-tuning)
#   - LR scheduler activado (ReduceLROnPlateau)
#   - class_weights activado (inversamente proporcional a frecuencia)
#   - epochs: 10 → 20
#   - patience: 3 → 5

project:
  name: "tesis-cxr-sprint2"
  sprint: 2
  seed: 42

paths:
  project_root: "/content/drive/MyDrive/Tesis_CXR"
  data_raw: "/content/drive/MyDrive/Tesis_CXR/data/raw/nih"
  data_processed: "/content/drive/MyDrive/Tesis_CXR/data/processed_s2"
  experiments: "/content/drive/MyDrive/Tesis_CXR/experiments_s2"
  checkpoints: "/content/drive/MyDrive/Tesis_CXR/experiments_s2/checkpoints"
  logs: "/content/drive/MyDrive/Tesis_CXR/experiments_s2/logs"

dataset:
  name: "nih"
  csv_filename: "Data_Entry_subset_local.csv"
  images_subdir: "images"
  views: ["PA", "AP"]

  # 4 clases con jerarquía de prioridad clínica
  # Si una imagen tiene múltiples labels, se asigna el de mayor prioridad.
  # Prioridad: Infiltration > Effusion > Cardiomegaly > No Finding
  classes:
    - name: "No Finding"
      label: 0
      priority: 0
    - name: "Cardiomegaly"
      label: 1
      priority: 1
    - name: "Effusion"
      label: 2
      priority: 2
    - name: "Infiltration"
      label: 3
      priority: 3   # prioridad más alta: TB/neumonía

  negative_class: "No Finding"

# Balance de clases (a nivel paciente, 1 img/paciente)
# ratio = n_negativos / n_positivos_por_clase
# Con ratio=2 y 3 clases positivas balanceadas a la más pequeña:
#   n_cardio = n_effusion = n_infiltration = min(820, ...)
#   n_no_finding = n_cardio * 2
class_ratio: 2.0

preprocessing:
  image_size: 224

splits:
  train_ratio: 0.70
  val_ratio: 0.15
  test_ratio: 0.15
  split_by: "patient_id"

model:
  backbone: "densenet121-res224-nih"
  num_classes: 4
  pretrained: true
  # "full": entrena TODOS los parámetros (6.9M)
  # Requiere LR más bajo para no destruir el preentrenamiento
  finetune_mode: "full"

training:
  batch_size: 32
  num_workers: 2
  epochs: 20
  learning_rate: 0.00001     # 10x más bajo que Sprint 1
  weight_decay: 0.0001
  optimizer: "adam"
  patience: 5                # más paciencia porque LR más bajo

  # LR scheduler: reduce LR si val loss no mejora
  scheduler:
    enabled: true
    type: "ReduceLROnPlateau"
    mode: "min"              # monitorear val loss
    factor: 0.5              # LR = LR * 0.5 cuando se activa
    patience: 2              # epochs sin mejora antes de reducir
    min_lr: 0.000001

  # Class weights: inversamente proporcionales a frecuencia
  # Evita que el modelo ignore las clases raras
  use_class_weights: true

  use_amp: true

evaluation:
  metrics: ["auc_macro", "auc_per_class", "accuracy", "precision_macro",
            "sensitivity_macro", "specificity_macro"]
  threshold: 0.5

gradcam:
  target_layer: "features.denseblock4"
  num_samples_per_class: 2   # 2 ejemplos por clase = 8 imágenes total
'''
path = f'{CODE_DIR}/configs/sprint2.yaml'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 4. Escribir `scripts/prepare_splits_s2.py`

In [ ]:
content = r'''"""
scripts/prepare_splits_s2.py

Genera los splits train/val/test para el Sprint 2: 4 clases.

Clases:
  0 - No Finding
  1 - Cardiomegaly   (insuficiencia cardíaca)
  2 - Effusion       (derrame pleural)
  3 - Infiltration   (TB pulmonar / neumonía)

Diferencias respecto a prepare_splits.py (Sprint 1):
  - 4 clases en vez de 2.
  - Asignación de labels por jerarquía de prioridad clínica.
    Una imagen con multiples patologias recibe el label de mayor prioridad:
    Infiltration (3) > Effusion (2) > Cardiomegaly (1) > No Finding (0).
  - Balance a nivel paciente: la clase mas pequeña define el techo.
    Todas las clases positivas se submuestrean a ese techo.
    No Finding se submuestrea a ratio*techo.
  - Misma estrategia de split v4: StratifiedKFold a nivel paciente
    + 1 imagen aleatoria por paciente.

Uso (desde Colab):
    !python /content/drive/MyDrive/Tesis_CXR/code/scripts/prepare_splits_s2.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config  # noqa: E402


# ---------------------------------------------------------------------------
# Clases y jerarquía
# ---------------------------------------------------------------------------
# Jerarquía de prioridad (mayor número = mayor prioridad).
# Si una imagen tiene múltiples patologías, se asigna la de mayor prioridad.
LABEL_HIERARCHY = {
    "Infiltration": 3,  # prioridad máxima: TB / neumonía
    "Effusion":     2,
    "Cardiomegaly": 1,
    "No Finding":   0,  # solo si no tiene ninguna de las anteriores
}

# Nombres de clase para logging
CLASS_NAMES = {0: "No Finding", 1: "Cardiomegaly",
               2: "Effusion",   3: "Infiltration"}


# ---------------------------------------------------------------------------
# Construcción de labels multi-clase con jerarquía
# ---------------------------------------------------------------------------
def build_multiclass_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Asigna un label entero 0-3 a cada imagen basándose en jerarquía clínica.

    Política:
      - Si 'Finding Labels' contiene 'Infiltration' → label=3
      - Si contiene 'Effusion'                      → label=2
      - Si contiene 'Cardiomegaly'                  → label=1
      - Si es exactamente 'No Finding'              → label=0
      - Cualquier otra combinación                  → NaN (se descarta)

    Nota: si una imagen tiene 'Effusion|Cardiomegaly', recibe label=2 (Effusion
    tiene prioridad). Esto refleja que en un contexto de triage el hallazgo
    de mayor urgencia define la clase.
    """
    df = df.copy()

    def assign_label(finding: str) -> float:
        for keyword, label in sorted(LABEL_HIERARCHY.items(),
                                     key=lambda x: -x[1]):
            if keyword == "No Finding":
                if finding == "No Finding":
                    return 0
            else:
                if keyword in finding:
                    return label
        return float('nan')  # ni "No Finding" ni ninguna de las 3 positivas

    df['label'] = df['Finding Labels'].apply(assign_label)
    return df


def filter_views(df: pd.DataFrame, views: list) -> pd.DataFrame:
    return df[df['View Position'].isin(views)].copy()


# ---------------------------------------------------------------------------
# Balance multi-clase a nivel paciente
# ---------------------------------------------------------------------------
def build_patient_level_label(df: pd.DataFrame) -> pd.Series:
    """
    Determina el label representativo de cada paciente usando la misma
    jerarquía: si un paciente tiene imágenes con distintos labels,
    se le asigna el de mayor prioridad.
    """
    return df.groupby('Patient ID')['label'].max().astype(int)


def balance_patients_multiclass(df: pd.DataFrame, ratio_negative: float,
                                seed: int, logger) -> pd.DataFrame:
    """
    Balancea el dataset a nivel paciente:
      1. Calcula el label representativo por paciente.
      2. Submuestrea cada clase positiva a la más pequeña (techo).
      3. Submuestrea No Finding a ratio_negative * techo.

    Args:
        df: DataFrame con columnas 'Patient ID', 'label'.
        ratio_negative: n_no_finding_pacientes = ratio_negative * n_min_positivo.
        seed: semilla para reproducibilidad.
    """
    rng = np.random.default_rng(seed)

    patient_labels = build_patient_level_label(df)

    # Separar pacientes por clase
    patients_by_class = {}
    for label in [0, 1, 2, 3]:
        pats = patient_labels[patient_labels == label].index.tolist()
        patients_by_class[label] = pats
        logger.info("  Clase %d (%s): %d pacientes",
                    label, CLASS_NAMES[label], len(pats))

    # Techo = mínimo entre las 3 clases positivas
    positive_counts = [len(patients_by_class[l]) for l in [1, 2, 3]]
    ceiling = min(positive_counts)
    logger.info("Techo de balance (clase más pequeña): %d pacientes", ceiling)

    # Submuestrear cada clase positiva al techo
    selected_patients = []
    for label in [1, 2, 3]:
        pats = patients_by_class[label]
        rng.shuffle(pats := list(pats))  # in-place shuffle
        selected = pats[:ceiling]
        selected_patients.extend(selected)
        logger.info("  Clase %d (%s): %d/%d pacientes seleccionados",
                    label, CLASS_NAMES[label], len(selected), len(pats))

    # Submuestrear No Finding
    n_neg_target = int(ceiling * ratio_negative)
    pats_neg = patients_by_class[0]
    rng.shuffle(pats_neg := list(pats_neg))
    selected_neg = pats_neg[:min(n_neg_target, len(pats_neg))]
    selected_patients.extend(selected_neg)
    logger.info("  Clase 0 (No Finding): %d pacientes seleccionados",
                len(selected_neg))

    # Filtrar df a los pacientes seleccionados
    selected_set = set(selected_patients)
    df_balanced = df[df['Patient ID'].isin(selected_set)].copy()
    df_balanced = df_balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    logger.info("Total tras balanceo: %d imágenes, %d pacientes",
                len(df_balanced), df_balanced['Patient ID'].nunique())
    return df_balanced


# ---------------------------------------------------------------------------
# Split estratificado a nivel paciente (misma lógica que Sprint 1 v4)
# ---------------------------------------------------------------------------
def stratified_patient_split(df: pd.DataFrame, train_ratio: float,
                             val_ratio: float, test_ratio: float,
                             seed: int, logger
                             ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    patient_labels = build_patient_level_label(df)
    logger.info("Pacientes únicos para split: %d", len(patient_labels))

    X_pat = patient_labels.index.to_numpy()
    y_pat = patient_labels.values

    # Paso 1: separar test
    n_splits_test = max(2, round(1.0 / test_ratio))
    skf_test = StratifiedKFold(n_splits=n_splits_test, shuffle=True,
                               random_state=seed)
    train_val_idx, test_idx = next(skf_test.split(X_pat, y_pat))

    test_patients = set(X_pat[test_idx])
    tv_patient_df = pd.DataFrame({
        'Patient ID': X_pat[train_val_idx],
        'patient_label': y_pat[train_val_idx],
    })

    # Paso 2: separar val de train+val
    val_ratio_rel = val_ratio / (train_ratio + val_ratio)
    n_splits_val = max(2, round(1.0 / val_ratio_rel))
    skf_val = StratifiedKFold(n_splits=n_splits_val, shuffle=True,
                              random_state=seed)
    X2 = tv_patient_df['Patient ID'].to_numpy()
    y2 = tv_patient_df['patient_label'].to_numpy()
    train_idx, val_idx = next(skf_val.split(X2, y2))

    train_patients = set(X2[train_idx])
    val_patients = set(X2[val_idx])

    logger.info("Split pacientes: train=%d, val=%d, test=%d",
                len(train_patients), len(val_patients), len(test_patients))

    df_train = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    df_val   = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
    df_test  = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)

    return df_train, df_val, df_test


def sample_one_image_per_patient(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    """Selecciona 1 imagen aleatoria por paciente (estrategia v4)."""
    rng = np.random.default_rng(seed)
    sampled = []
    for _, group in df.groupby('Patient ID'):
        idx = rng.integers(0, len(group))
        sampled.append(group.iloc[idx])
    return pd.DataFrame(sampled).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Reporte y validaciones
# ---------------------------------------------------------------------------
def report_split(name: str, df: pd.DataFrame, logger) -> None:
    n = len(df)
    n_patients = df['Patient ID'].nunique()
    logger.info("%-5s: %4d imgs | %3d pacientes", name.upper(), n, n_patients)
    for label, cname in CLASS_NAMES.items():
        count = (df['label'] == label).sum()
        pct = count / n * 100 if n else 0
        logger.info("  clase %d (%s): %d (%.1f%%)", label, cname, count, pct)


def check_no_leakage(train, val, test, logger) -> None:
    p_tr = set(train['Patient ID'])
    p_va = set(val['Patient ID'])
    p_te = set(test['Patient ID'])
    for name, overlap in [('train-val', p_tr & p_va),
                           ('train-test', p_tr & p_te),
                           ('val-test', p_va & p_te)]:
        if overlap:
            logger.error("LEAKAGE en %s: %d pacientes", name, len(overlap))
            raise AssertionError(f"Data leakage en {name}")
    logger.info("Sin leakage: pacientes disjuntos entre splits.")


def check_stratification(train, val, test, logger,
                         max_deviation: float = 10.0) -> None:
    combined = pd.concat([train, val, test], ignore_index=True)
    logger.info("Distribución global de clases:")
    for label, cname in CLASS_NAMES.items():
        global_pct = (combined['label'] == label).mean() * 100
        logger.info("  clase %d (%s): %.1f%% global", label, cname, global_pct)
        for sname, sdf in [("train", train), ("val", val), ("test", test)]:
            pct = (sdf['label'] == label).mean() * 100
            dev = abs(pct - global_pct)
            if dev > max_deviation:
                logger.warning("  %s tiene %.1f%% de clase %d "
                               "(desviación %.1fpp)", sname, pct, label, dev)
    logger.info("Verificación de estratificación completada.")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "sprint2.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])
    log_file = Path(cfg.paths['logs']) / "prepare_splits_s2.log"
    logger = get_logger("prepare_splits_s2", log_file=log_file)

    logger.info("=" * 60)
    logger.info("PREPARE SPLITS Sprint 2 — 4 clases")
    logger.info("Jerarquía: Infiltration(3) > Effusion(2) > "
                "Cardiomegaly(1) > No Finding(0)")
    logger.info("=" * 60)

    nih_dir = Path(cfg.paths['data_raw'])
    processed_dir = Path(cfg.paths['data_processed'])
    processed_dir.mkdir(parents=True, exist_ok=True)

    # 1. Cargar CSV combinado de los 3 tarballs
    subset_csv = nih_dir / "Data_Entry_subset_local.csv"
    if not subset_csv.exists():
        raise FileNotFoundError(
            f"No se encontró {subset_csv}. "
            "Corre primero S2_01_download_more_data.ipynb."
        )

    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar vistas
    df = filter_views(df, cfg.dataset['views'])
    logger.info("Tras filtrar vistas %s: %d filas",
                cfg.dataset['views'], len(df))

    # 3. Construir labels con jerarquía
    df = build_multiclass_labels(df)
    n_before = len(df)
    df = df.dropna(subset=['label']).copy()
    df['label'] = df['label'].astype(int)
    logger.info("Tras construir labels: %d filas (descartadas %d sin clase)",
                len(df), n_before - len(df))

    for label, cname in CLASS_NAMES.items():
        n = (df['label'] == label).sum()
        logger.info("  %s: %d imágenes", cname, n)

    # 4. Balancear a nivel paciente
    ratio = cfg.get('class_ratio', 2.0)
    logger.info("Balanceando con ratio NF=%.1f * min_positivo...", ratio)
    df_balanced = balance_patients_multiclass(
        df, ratio_negative=ratio, seed=cfg.project['seed'], logger=logger
    )

    # 5. Split estratificado por paciente
    train_df, val_df, test_df = stratified_patient_split(
        df_balanced,
        train_ratio=cfg.splits['train_ratio'],
        val_ratio=cfg.splits['val_ratio'],
        test_ratio=cfg.splits['test_ratio'],
        seed=cfg.project['seed'],
        logger=logger,
    )

    # 6. Una imagen por paciente
    train_df = sample_one_image_per_patient(train_df, seed=cfg.project['seed'])
    val_df   = sample_one_image_per_patient(val_df,   seed=cfg.project['seed'] + 1)
    test_df  = sample_one_image_per_patient(test_df,  seed=cfg.project['seed'] + 2)

    # 7. Reportar y validar
    logger.info("-" * 60)
    report_split("train", train_df, logger)
    report_split("val",   val_df,   logger)
    report_split("test",  test_df,  logger)
    logger.info("-" * 60)
    check_no_leakage(train_df, val_df, test_df, logger)
    check_stratification(train_df, val_df, test_df, logger)

    # 8. Guardar
    cols = ['Image Index', 'Finding Labels', 'Patient ID', 'View Position', 'label']
    train_df[cols].to_csv(processed_dir / "train.csv", index=False)
    val_df[cols].to_csv(processed_dir   / "val.csv",   index=False)
    test_df[cols].to_csv(processed_dir  / "test.csv",  index=False)

    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(train_df))
    logger.info("  val.csv:   %d filas", len(val_df))
    logger.info("  test.csv:  %d filas", len(test_df))
    logger.info("=" * 60)
    logger.info("PREPARE SPLITS S2 OK")


if __name__ == "__main__":
    main()
'''
path = f'{CODE_DIR}/scripts/prepare_splits_s2.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 5. Ejecutar el script de splits Sprint 2

Los splits se guardan en `data/processed_s2/` (carpeta nueva, no
sobreescribe los splits del Sprint 1).

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits_s2.py'],
    capture_output=True, text=True,
)
print('STDOUT:')
print(result.stdout)
print()
print('STDERR (log):')
print(result.stderr)


## 6. Verificar los splits generados

In [ ]:
import pandas as pd

processed_s2 = f'{PROJECT_ROOT}/data/processed_s2'
CLASS_NAMES = {0: 'No Finding', 1: 'Cardiomegaly',
               2: 'Effusion',   3: 'Infiltration'}

print('RESUMEN SPLITS SPRINT 2:')
print('=' * 70)

total_pos, total_neg = 0, 0
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed_s2}/{split}.csv')
    n_pat = df['Patient ID'].nunique()
    print(f'\n{split.upper()} ({len(df)} imgs | {n_pat} pacientes):')
    for label, cname in CLASS_NAMES.items():
        n = (df['label'] == label).sum()
        pct = n / len(df) * 100
        bar = '█' * int(pct / 2)
        print(f'  {label} {cname:<15}: {n:>4} ({pct:4.1f}%) {bar}')

print()
print('COMPARATIVA SPRINT 1 vs SPRINT 2:')
print(f'  Sprint 1: 279 imgs, 2 clases, mode=head')
print(f'  Sprint 2: ~{sum(len(pd.read_csv(f"{processed_s2}/{s}.csv")) for s in ["train","val","test"])} imgs, 4 clases, mode=full')


## 7. Siguiente paso

Si los 4 splits muestran distribuciones balanceadas (~25% por clase),
los datos están listos para entrenamiento.

Vuelve al chat para recibir:
1. `src/training/train.py` actualizado (LR scheduler + class weights).
2. `src/evaluation/metrics.py` actualizado (AUC por clase + macro).
3. `scripts/run_train_s2.py` — entrypoint Sprint 2.
